# Institutional Quant AI v6.0: Temporal Fusion Engine

Brief sub-header: A multi-asset neural forging and backtesting pipeline built on Keras 3 and NVIDIA G4/L4 hardware.

## Phase 0: Environment Hardware Initialization

Purpose: Upgrading the runtime to Keras 3 Native and establishing a high-performance memory allocator for the RTX 6000.

In [ ]:
# ==============================================================================
# CELL 0: DEPENDENCIES & ENVIRONMENT SETUP (KERAS 3 NATIVE)
# RUNTIME: General
# PURPOSE: Installs latest TensorFlow (Keras 3) and data science tools.
# ==============================================================================
import os
if "TF_USE_LEGACY_KERAS" in os.environ:
    del os.environ["TF_USE_LEGACY_KERAS"]

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

print("[SYSTEM] Updating to Keras 3 native environment...")

!pip install -q --upgrade tensorflow optuna numba yfinance scikit-learn transformers datasets torch streamlit pydeck colorlog > /dev/null 2>&1

print("[SYSTEM] Environment Restored. Ready for Native Keras 3 Forge.")

## Phase 1: Persistent Data Bridge (Google Drive)

Purpose: Mounting the cloud filesystem to secure model weights and historical hyperparameter manifests.

In [ ]:
# ==============================================================================
# CELL 1: GOOGLE DRIVE MOUNT
# RUNTIME: General (Run once globally on CPU, T4, or G4)
# PURPOSE: Bridges the Colab instance to your persistent Drive storage.
# ==============================================================================
import os
from google.colab import drive

drive.mount('/content/drive')
PROJECT_PATH = '/content/drive/MyDrive/sentiment_csv'

if os.path.exists(PROJECT_PATH):
    print(f"Success! Connected to: {PROJECT_PATH}")
    print(f"Files found: {os.listdir(PROJECT_PATH)}")

## Phase 2: Natural Language Processing (NLP) Ingestion

Purpose: Leveraging FinBERT to extract daily sentiment scores from high-frequency financial news streams.

In [ ]:
# ==============================================================================
# CELL 2: FINBERT SENTIMENT PIPELINE
# RUNTIME: T4 Free GPU
# PURPOSE: Scrapes NLP sentiment. Run ONLY if you need to update sentiment data.
# ==============================================================================
import pandas as pd
import torch
import gc
from datasets import load_dataset
from transformers import pipeline
from google.colab import userdata

# ==============================================================================
# 1. SECURE CONFIGURATION & LOCAL NLP INITIALIZATION
# ==============================================================================
try:
    HF_TOKEN = userdata.get('HF_TOKEN')
except userdata.SecretNotFoundError:
    print("[CRITICAL ERROR] Secret 'HF_TOKEN' not found. Please add it to Colab Secrets.")
    raise

# NOTE: Adjust this list! If you only need Apple right now, change to: TICKERS = ["AAPL"]
TICKERS = ["ETH-USD"]

# Institutional Backtest Target: ~10 years of data
MAX_ARTICLES_PER_ASSET = 10000

device = 0 if torch.cuda.is_available() else -1
if device == 0:
    print(" GPU Detected! FinBERT will run with hardware acceleration.")
else:
    print(" No GPU detected. FinBERT will run on CPU (slower).")

print("Loading FinBERT Financial NLP Model into local memory...")
sentiment_analyzer = pipeline("sentiment-analysis", model="ProsusAI/finbert", device=device)
print("FinBERT Model Loaded Successfully!")

# ==============================================================================
# 2. FINBERT REASONING ENGINE (UPGRADED WITH GPU BATCHING)
# ==============================================================================
def analyze_daily_news(headlines_list):
    """Scores headlines using GPU Batching (batch_size=16) for maximum speed."""
    if not headlines_list:
        return 0.0

    # Truncate to 512 chars to prevent model indexing errors on massive articles
    safe_headlines = [str(h)[:512] for h in headlines_list]

    daily_score = 0.0
    valid_articles = 0

    try:
        # THE BATCHING OPTIMIZATION: Process 16 articles simultaneously on the GPU
        results = sentiment_analyzer(safe_headlines, batch_size=16)

        for result in results:
            label = result['label']
            if label == 'positive':
                daily_score += 1.0
            elif label == 'negative':
                daily_score += -1.0
            else:
                daily_score += 0.0

            valid_articles += 1

    except Exception as e:
        # If a severely corrupted batch crashes the pipeline, skip it safely
        return 0.0

    if valid_articles > 0:
        return round(daily_score / valid_articles, 4)
    return 0.0

# ==============================================================================
# 3. FAULT-TOLERANT DATA STREAMING PIPELINE (WITH CIRCUIT BREAKER)
# ==============================================================================
def fetch_raw_historical_news(ticker):
    print(f"\nStreaming Massive Hugging Face dataset for {ticker}...")

    ASSET_NAMES = {
        "AAPL": "apple",
        "NVDA": "nvidia",
        "MSFT": "microsoft",
        "GOOGL": "google",
        "SPY": "s&p 500",
        "BTC-USD": "bitcoin",
        "ETH-USD": "ethereum"
    }

    search_name = ASSET_NAMES.get(ticker, ticker.lower())

    try:
        dataset = load_dataset(
            "Brianferrell787/financial-news-multisource",
            split="train",
            streaming=True,
            token=HF_TOKEN
        )
    except Exception as e:
        print(f"[!] HuggingFace connection error: {e}")
        return {}

    news_dict = {}
    extracted_count = 0
    rows_scanned = 0

    # NEW: THE CIRCUIT BREAKER
    # If the script reads 3 million rows, it forces a stop to prevent RAM crashes.
    MAX_SCAN_DEPTH = 3000000

    try:
        for row in dataset:
            rows_scanned += 1
            text = str(row.get('text', ''))
            extra = str(row.get('extra_fields', ''))

            if search_name in text.lower() or ticker in extra:
                raw_date = row.get('date')
                try:
                    clean_date = pd.to_datetime(raw_date).strftime('%Y-%m-%d')
                except:
                    continue

                if clean_date not in news_dict:
                    news_dict[clean_date] = []

                news_dict[clean_date].append(text)
                extracted_count += 1

                if extracted_count % 500 == 0:
                    print(f"   ...Found {extracted_count}/{MAX_ARTICLES_PER_ASSET} articles for {ticker}...")

                if extracted_count >= MAX_ARTICLES_PER_ASSET:
                    break

            # The Circuit Breaker Trigger
            if rows_scanned >= MAX_SCAN_DEPTH:
                print(f"\n   [!] WARNING: Scanned {MAX_SCAN_DEPTH} rows. Stopping search to protect Colab RAM.")
                break

    except Exception as e:
        print(f"\n   [!] HUGGING FACE SERVER CRASH INTERCEPTED: {e}")

    print(f"   -> Saving the {extracted_count} articles we successfully grabbed.")
    print(f"[SUCCESS] Extracted {extracted_count} articles across {len(news_dict)} trading days for {ticker}.")
    return news_dict

# ==============================================================================
# 4. MEMORY-SAFE BATCH EXECUTION
# ==============================================================================
def run_sentiment_miner():
    print("\n--- STARTING OPTIMIZED NLP PIPELINE ---")

    for ticker in TICKERS:
        print(f"\n================ Processing {ticker} ================")
        historical_news = fetch_raw_historical_news(ticker)

        results = []
        total_days = len(historical_news)

        if total_days == 0:
            print(f" [WARNING] No data found for {ticker}, skipping.")
            continue

        print(f"[{ticker}] Scoring {total_days} days using GPU Batching... Please wait.")

        for date, headlines in historical_news.items():
            score = analyze_daily_news(headlines)
            results.append({"Date": date, "Sentiment_Score": score})

        df_results = pd.DataFrame(results)
        df_results['Date'] = pd.to_datetime(df_results['Date'])
        df_results.sort_values('Date', inplace=True)

        output_filename = f"{ticker}_sentiment.csv"
        df_results.to_csv(output_filename, index=False)
        print(f" [SAVED] {output_filename} is ready! (Total days scored: {total_days})")

        # ----------------------------------------------------------------------
        # GARBAGE COLLECTION TO PREVENT RAM CRASHES
        # ----------------------------------------------------------------------
        print(f"🧹 Sweeping memory logs to protect Colab RAM...")
        del historical_news
        del results
        del df_results
        gc.collect()

        if torch.cuda.is_available():
            torch.cuda.empty_cache()
        # ----------------------------------------------------------------------

run_sentiment_miner()

## Phase 3: Research Archive & T4 Benchmarking

Note: These cells are preserved for performance comparison between T4 (Free) and G4 (Pro) runtimes. They represent the legacy Keras 2 implementation.

In [ ]:
# ==============================================================================
# CELL 3: T4 LITE FORGE (SANDBOX)
# RUNTIME: T4 Free GPU
# ==============================================================================
import optuna
from optuna.integration import TFKerasPruningCallback
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Bidirectional, Conv1D, GRU, LSTM,
    BatchNormalization, GaussianNoise, LayerNormalization, Add,
    MultiHeadAttention, GlobalAveragePooling1D, Multiply, Lambda, Reshape, Concatenate
)
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
from tensorflow.keras.regularizers import l2
import shutil
import warnings
import os
import gc

tf.random.set_seed(42)
np.random.seed(42)
mixed_precision.set_global_policy('mixed_float16')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

TICKERS = ["AAPL", "BTC-USD"]
PROJECT_PATH = '/content/drive/MyDrive/sentiment_csv'
DB_PATH = "/content/t4_optuna_test.db"
DRIVE_DB_PATH = f"{PROJECT_PATH}/t4_optuna_test.db"

ARCHS = [
    "TCN + MHA + GRU (Sniper Alpha v2)",
    "BiLSTM + Attention (Deep Memory v2)"
]

def sharpe_loss(y_true, y_pred):
    strategy_returns = y_pred * y_true
    mean_return = K.mean(strategy_returns)
    std_return = K.std(strategy_returns) + 1e-9
    return -(mean_return / std_return)

class Time2Vec(tf.keras.layers.Layer):
    def __init__(self, kernel_size=1, **kwargs):
        super(Time2Vec, self).__init__(**kwargs)
        self.k = kernel_size

    def build(self, input_shape):
        self.wb = self.add_weight(name='wb', shape=(input_shape[-1],), initializer='uniform', trainable=True)
        self.bb = self.add_weight(name='bb', shape=(input_shape[-1],), initializer='uniform', trainable=True)
        self.wa = self.add_weight(name='wa', shape=(1, input_shape[-1], self.k), initializer='uniform', trainable=True)
        self.ba = self.add_weight(name='ba', shape=(1, input_shape[-1], self.k), initializer='uniform', trainable=True)
        super(Time2Vec, self).build(input_shape)

    def call(self, inputs):
        bias = self.wb * inputs + self.bb
        dp = K.dot(inputs, self.wa) + self.ba
        wgts = tf.math.sin(dp)
        ret = tf.concat([tf.expand_dims(bias, -1), wgts], -1)
        ret = tf.reshape(ret, (-1, tf.shape(inputs)[1], ret.shape[2] * ret.shape[3]))
        return ret

def build_t4_model(arch, input_shape, num_features, dropout_rate):
    inputs = Input(shape=input_shape)
    x = GaussianNoise(0.005)(inputs)
    x = BatchNormalization()(x)
    time_embedding = Time2Vec(kernel_size=4)(x)
    x = Concatenate()([x, time_embedding])

    if arch == "TCN + MHA + GRU (Sniper Alpha v2)":
        c1 = Conv1D(64, 3, dilation_rate=1, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(x)
        c8 = Conv1D(64, 3, dilation_rate=8, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(
             Conv1D(64, 3, dilation_rate=4, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(
             Conv1D(64, 3, dilation_rate=2, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c1)))
        tcn = LayerNormalization()(Add()([c1, c8]))
        attn = LayerNormalization()(Add()([tcn, MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(tcn, tcn)]))
        x = Dropout(dropout_rate)(Bidirectional(GRU(128, return_sequences=False))(attn))

    elif arch == "BiLSTM + Attention (Deep Memory v2)":
        x = Dropout(dropout_rate)(LayerNormalization()(Bidirectional(LSTM(128, return_sequences=True))(x)))
        x = Dropout(dropout_rate)(LayerNormalization()(Bidirectional(LSTM(64, return_sequences=True))(x)))
        score = Dense(1)(x)
        weights_attn = tf.keras.layers.Softmax(axis=1)(score)
        x = Lambda(lambda z: tf.reduce_sum(z, axis=1))(Multiply()([x, weights_attn]))

    x = Dense(128, activation='gelu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)
    # FIX: Tanh bounds predictions directly mapping to position weights [-1, 1], stabilizing Sharpe Loss
    outputs = Dense(1, activation='tanh', dtype='float32')(x)
    return Model(inputs, outputs)

print("\n========================================================")
print(" INITIATING T4 LITE FORGE (TEST MODE)")
print("========================================================\n")

for stock in TICKERS:
    print(f"\n---> Forging Asset: {stock}")
    start_date = (pd.to_datetime("today") - pd.DateOffset(days=720)).strftime('%Y-%m-%d')
    end_date = pd.to_datetime("today").strftime('%Y-%m-%d')

    try:
        df = yf.download(stock, start=start_date, end=end_date, interval="1h", progress=False)
        macro_tickers = ["^VIX", "^TNX", "SPY", "BTC-USD"]
        macro_df = yf.download(macro_tickers, start=start_date, end=end_date, interval="1h", progress=False)

        if isinstance(macro_df.columns, pd.MultiIndex): macro_df = macro_df['Close']
        macro_df.rename(columns={'^VIX': 'VIX', '^TNX': 'TNX'}, inplace=True)
        macro_df.ffill(inplace=True)

        macro_df['SPY_Ret'] = np.log(macro_df['SPY'] / macro_df['SPY'].shift(1))
        macro_df['BTC_Ret'] = np.log(macro_df['BTC-USD'] / macro_df['BTC-USD'].shift(1))
        macro_df.drop(columns=['SPY', 'BTC-USD'], inplace=True, errors='ignore')

        if type(macro_df.index) == pd.DatetimeIndex: macro_df.index = macro_df.index.tz_localize(None)
        if 'Date' not in df.columns: df = df.reset_index()
        df.rename(columns={'Datetime':'Date', 'date':'Date', 'time':'Date', 'index':'Date'}, inplace=True, errors='ignore')
        df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)

        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        if 'Adj Close' in df.columns and 'Close' not in df.columns: df['Close'] = df['Adj Close']

        df = pd.merge(df, macro_df, left_on='Date', right_index=True, how='left').ffill()
        df['Sentiment_Score'] = 0.0
        df.set_index('Date', inplace=True)

        c = df['Close']
        df['Log_Returns'] = np.log(c / c.shift(1))

        # Target calculation (6 hours ahead)
        df['Target'] = (df['Close'].shift(-6) / df['Close']) - 1.0

        h, l, o = df['High'], df['Low'], df['Open']
        df['Garman_Klass'] = (0.5 * np.log(h/l)**2 - (2*np.log(2)-1) * np.log(c/o)**2).clip(lower=0)
        df['Garman_Klass'] = df['Garman_Klass'].ewm(span=72, adjust=False).mean()

        df['RSI'] = 100 - 100 / (1 + c.diff().where(c.diff()>0,0).rolling(14).mean() / (-c.diff().where(c.diff()<0,0)).rolling(14).mean().replace(0,1e-9))
        df['MACD'] = c.ewm(span=12).mean() - c.ewm(span=26).mean()

        is_crypto = stock in ["BTC-USD", "ETH-USD"]
        ann_factor = 8760 if is_crypto else 1638
        df['Vol_Regime'] = df['Log_Returns'].rolling(20).std() * np.sqrt(ann_factor)

        t = df.index
        is_weekday = t.dayofweek < 5
        is_trading_hours = (t.hour >= 9) & (t.hour <= 16)
        df['RTH_Active'] = (is_weekday & is_trading_hours).astype(float)

        features = ['Log_Returns', 'RSI', 'MACD', 'Vol_Regime', 'VIX', 'TNX', 'SPY_Ret', 'BTC_Ret', 'Sentiment_Score', 'RTH_Active']

        # FIX: Drop NaNs ONLY based on features, preserving the final 6 rows where Target is NaN
        df.dropna(subset=features, inplace=True)

    except Exception as e:
        print(f"[!] Data failure for {stock}: {e}. Skipping.")
        continue

    def objective(trial):
        tf.keras.backend.clear_session()
        gc.collect()

        arch = trial.suggest_categorical("arch", ARCHS)
        ts = trial.suggest_int("time_step", 24, 72, step=24)
        lr = trial.suggest_float("lr", 1e-4, 1e-3, log=True)
        drop = trial.suggest_float("dropout", 0.2, 0.5)
        batch = trial.suggest_categorical("batch", [64, 128, 256])

        split = int(len(df) * 0.8)
        train, test = df.iloc[:split], df.iloc[split:]

        scaler = RobustScaler()
        X_tr_sc = scaler.fit_transform(train[features]).astype(np.float32)
        X_te_sc = scaler.transform(test[features]).astype(np.float32)

        y_tr = train['Target'].values.astype(np.float32)
        y_te = test['Target'].values.astype(np.float32)

        def window(X, y, t):
            limit = len(X) - t + 1
            Xs = np.array([X[i:i+t] for i in range(limit)])
            ys = np.array([y[i+t-1] for i in range(limit)])
            return Xs, ys

        X_tr, y_tr = window(X_tr_sc, y_tr, ts)
        X_te, y_te = window(X_te_sc, y_te, ts)

        # Mask out Target NaNs for training/validation explicitly
        valid_tr = ~np.isnan(y_tr)
        X_tr, y_tr = X_tr[valid_tr], y_tr[valid_tr]

        valid_te = ~np.isnan(y_te)
        X_te_eval, y_te_eval = X_te[valid_te], y_te[valid_te]

        model = build_t4_model(arch, (X_tr.shape[1], X_tr.shape[2]), len(features), drop)
        model.compile(optimizer=Adam(learning_rate=lr, clipnorm=1.0), loss=sharpe_loss)

        model.fit(
            X_tr, y_tr, validation_data=(X_te_eval, y_te_eval),
            epochs=10, batch_size=batch,
            callbacks=[EarlyStopping(patience=3), TFKerasPruningCallback(trial, 'val_loss')],
            verbose=0
        )

        return float(model.evaluate(X_te_eval, y_te_eval, verbose=0))

    study_name = f"t4_test_{stock}"
    study = optuna.create_study(
        study_name=study_name,
        storage=f"sqlite:///{DB_PATH}",
        load_if_exists=True,
        direction="minimize"
    )

    print(f"[{stock}] Testing loop logic (10 Trials)...")
    study.optimize(objective, n_trials=10)
    print(f"[{stock}] Test Complete. Best Arch: {study.best_params['arch']}")

    try:
        shutil.copy(DB_PATH, DRIVE_DB_PATH)
        print(f"[{stock}] Database securely backed up to Drive.")
    except Exception as e:
        print(f"[-] Backup failed: {e}")

print("\n========================================================")
print(" T4 TEST FORGE COMPLETE.")
print("========================================================")

In [ ]:
# ==============================================================================
# CELL 3.5: T4 MAXIMUM OVERDRIVE FORGE
# RUNTIME: T4 Free GPU
# PURPOSE: Pushes the absolute limit of the free 16GB VRAM without OOM crashing.
# ==============================================================================
import optuna
from optuna.integration import TFKerasPruningCallback
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Dropout, Input, Bidirectional, Conv1D, GRU,
    BatchNormalization, GaussianNoise, LayerNormalization, Add,
    MultiHeadAttention, Multiply, Lambda, Concatenate
)
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras import mixed_precision
from tensorflow.keras.regularizers import l2
import shutil
import warnings
import os
import gc

tf.random.set_seed(42)
np.random.seed(42)
mixed_precision.set_global_policy('mixed_float16')
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"
warnings.filterwarnings("ignore")
optuna.logging.set_verbosity(optuna.logging.WARNING)

TICKERS = ["AAPL", "NVDA", "BTC-USD"]
PROJECT_PATH = '/content/drive/MyDrive/sentiment_csv'
DB_PATH = "/content/t4_max_forge.db"
DRIVE_DB_PATH = f"{PROJECT_PATH}/t4_max_forge.db"

def sharpe_loss(y_true, y_pred):
    strategy_returns = y_pred * y_true
    mean_return = K.mean(strategy_returns)
    std_return = K.std(strategy_returns) + 1e-9
    return -(mean_return / std_return)

class Time2Vec(tf.keras.layers.Layer):
    def __init__(self, kernel_size=1, **kwargs):
        super(Time2Vec, self).__init__(**kwargs)
        self.k = kernel_size

    def build(self, input_shape):
        self.wb = self.add_weight(name='wb', shape=(input_shape[-1],), initializer='uniform', trainable=True)
        self.bb = self.add_weight(name='bb', shape=(input_shape[-1],), initializer='uniform', trainable=True)
        self.wa = self.add_weight(name='wa', shape=(1, input_shape[-1], self.k), initializer='uniform', trainable=True)
        self.ba = self.add_weight(name='ba', shape=(1, input_shape[-1], self.k), initializer='uniform', trainable=True)
        super(Time2Vec, self).build(input_shape)

    def call(self, inputs):
        bias = self.wb * inputs + self.bb
        dp = K.dot(inputs, self.wa) + self.ba
        wgts = tf.math.sin(dp)
        ret = tf.concat([tf.expand_dims(bias, -1), wgts], -1)
        ret = tf.reshape(ret, (-1, tf.shape(inputs)[1], ret.shape[2] * ret.shape[3]))
        return ret

def build_t4_max_model(input_shape, dropout_rate):
    inputs = Input(shape=input_shape)
    x = GaussianNoise(0.005)(inputs)
    x = BatchNormalization()(x)

    time_embedding = Time2Vec(kernel_size=4)(x)
    x = Concatenate()([x, time_embedding])

    # 1. High-Speed Temporal Feature Extraction (TCN)
    c1 = Conv1D(64, 3, dilation_rate=1, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(x)
    c2 = Conv1D(64, 3, dilation_rate=2, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c1)
    c4 = Conv1D(64, 3, dilation_rate=4, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c2)

    # Residual projection to match dimensions
    res = Conv1D(64, 1, padding='same')(x)
    tcn = LayerNormalization()(Add()([res, c4]))

    # 2. Lightweight Attention (Constrained for 16GB VRAM)
    attn = MultiHeadAttention(num_heads=2, key_dim=32, dropout=0.1)(tcn, tcn)
    x = LayerNormalization()(Add()([tcn, attn]))

    # 3. Truncated Recurrence (GRU is faster than LSTM on T4)
    x = Dropout(dropout_rate)(Bidirectional(GRU(128, return_sequences=False))(x))

    # 4. Dense Head with Bounded Output
    x = Dense(64, activation='gelu', kernel_regularizer=l2(1e-4))(x)
    x = Dropout(0.3)(x)
    outputs = Dense(1, activation='tanh', dtype='float32')(x)

    return Model(inputs, outputs)

print("\n========================================================")
print(" INITIATING T4 MAXIMUM OVERDRIVE FORGE")
print("========================================================\n")

for stock in TICKERS:
    print(f"\n---> Forging Asset: {stock}")

    start_date = (pd.to_datetime("today") - pd.DateOffset(days=720)).strftime('%Y-%m-%d')
    end_date = pd.to_datetime("today").strftime('%Y-%m-%d')

    try:
        df = yf.download(stock, start=start_date, end=end_date, interval="1h", progress=False)
        macro_tickers = ["^VIX", "^TNX", "SPY", "BTC-USD"]
        macro_df = yf.download(macro_tickers, start=start_date, end=end_date, interval="1h", progress=False)

        if isinstance(macro_df.columns, pd.MultiIndex): macro_df = macro_df['Close']
        macro_df.rename(columns={'^VIX': 'VIX', '^TNX': 'TNX'}, inplace=True)
        macro_df.ffill(inplace=True)

        macro_df['SPY_Ret'] = np.log(macro_df['SPY'] / macro_df['SPY'].shift(1))
        macro_df['BTC_Ret'] = np.log(macro_df['BTC-USD'] / macro_df['BTC-USD'].shift(1))
        macro_df.drop(columns=['SPY', 'BTC-USD'], inplace=True, errors='ignore')

        if type(macro_df.index) == pd.DatetimeIndex: macro_df.index = macro_df.index.tz_localize(None)
        if 'Date' not in df.columns: df = df.reset_index()
        df.rename(columns={'Datetime':'Date', 'date':'Date', 'time':'Date', 'index':'Date'}, inplace=True, errors='ignore')
        df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)

        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
        if 'Adj Close' in df.columns and 'Close' not in df.columns: df['Close'] = df['Adj Close']

        df = pd.merge(df, macro_df, left_on='Date', right_index=True, how='left').ffill()
        df['Sentiment_Score'] = 0.0
        df.set_index('Date', inplace=True)

        c = df['Close']
        df['Log_Returns'] = np.log(c / c.shift(1))
        df['Target'] = (df['Close'].shift(-6) / df['Close']) - 1.0

        h, l, o = df['High'], df['Low'], df['Open']
        df['Garman_Klass'] = (0.5 * np.log(h/l)**2 - (2*np.log(2)-1) * np.log(c/o)**2).clip(lower=0)
        df['Garman_Klass'] = df['Garman_Klass'].ewm(span=72, adjust=False).mean()

        df['RSI'] = 100 - 100 / (1 + c.diff().where(c.diff()>0,0).rolling(14).mean() / (-c.diff().where(c.diff()<0,0)).rolling(14).mean().replace(0,1e-9))
        df['MACD'] = c.ewm(span=12).mean() - c.ewm(span=26).mean()

        is_crypto = stock in ["BTC-USD", "ETH-USD"]
        ann_factor = 8760 if is_crypto else 1638
        df['Vol_Regime'] = df['Log_Returns'].rolling(20).std() * np.sqrt(ann_factor)

        t = df.index
        is_weekday = t.dayofweek < 5
        is_trading_hours = (t.hour >= 9) & (t.hour <= 16)
        df['RTH_Active'] = (is_weekday & is_trading_hours).astype(float)

        features = ['Log_Returns', 'RSI', 'MACD', 'Vol_Regime', 'VIX', 'TNX', 'SPY_Ret', 'BTC_Ret', 'Sentiment_Score', 'RTH_Active']
        df.dropna(subset=features, inplace=True)

    except Exception as e:
        print(f"[!] Data failure for {stock}: {e}. Skipping.")
        continue

    def objective(trial):
        tf.keras.backend.clear_session()
        gc.collect()

        ts = trial.suggest_int("time_step", 48, 96, step=24)
        lr = trial.suggest_float("lr", 1e-4, 8e-4, log=True)
        drop = trial.suggest_float("dropout", 0.25, 0.45)
        batch = trial.suggest_categorical("batch", [256, 512])

        split = int(len(df) * 0.8)
        train, test = df.iloc[:split], df.iloc[split:]

        scaler = RobustScaler()
        X_tr_sc = scaler.fit_transform(train[features]).astype(np.float32)
        X_te_sc = scaler.transform(test[features]).astype(np.float32)

        y_tr = train['Target'].values.astype(np.float32)
        y_te = test['Target'].values.astype(np.float32)

        def window(X, y, t):
            limit = len(X) - t + 1
            Xs = np.array([X[i:i+t] for i in range(limit)])
            ys = np.array([y[i+t-1] for i in range(limit)])
            return Xs, ys

        X_tr, y_tr = window(X_tr_sc, y_tr, ts)
        X_te, y_te = window(X_te_sc, y_te, ts)

        valid_tr = ~np.isnan(y_tr)
        X_tr, y_tr = X_tr[valid_tr], y_tr[valid_tr]

        valid_te = ~np.isnan(y_te)
        X_te_eval, y_te_eval = X_te[valid_te], y_te[valid_te]

        # Explicit strategy declaration to flush memory properly on T4
        strategy = tf.distribute.MirroredStrategy()
        with strategy.scope():
            model = build_t4_max_model((X_tr.shape[1], X_tr.shape[2]), drop)
            model.compile(optimizer=Adam(learning_rate=lr, clipnorm=1.0), loss=sharpe_loss)

        model.fit(
            X_tr, y_tr, validation_data=(X_te_eval, y_te_eval),
            epochs=15, batch_size=batch,
            callbacks=[EarlyStopping(patience=4), TFKerasPruningCallback(trial, 'val_loss')],
            verbose=0
        )

        return float(model.evaluate(X_te_eval, y_te_eval, verbose=0))

    study_name = f"t4_max_{stock}"
    study = optuna.create_study(study_name=study_name, storage=f"sqlite:///{DB_PATH}", load_if_exists=True, direction="minimize")

    # 15 Trials strikes the exact balance between finding edge and hitting Colab's time limits
    print(f"[{stock}] Hunting T4 hyper-space (15 Trials)...")
    study.optimize(objective, n_trials=15)

    print(f"[{stock}] Forge Complete. Best Time Step: {study.best_params['time_step']}")

    try:
        shutil.copy(DB_PATH, DRIVE_DB_PATH)
        print(f"[{stock}] Database securely backed up to Drive.")
    except Exception as e:
        print(f"[-] Backup failed: {e}")

print("\n========================================================")
print(" T4 MAXIMUM OVERDRIVE FORGE COMPLETE.")
print("========================================================")

## Phase 4: The Production Forge (Temporal Fusion Transformer)

Purpose: This is the core neural engine. It implements Variable Selection Networks (VSN) and Multi-Head Attention to generate predictive alpha. Outputs are saved as .weights.h5 and .json manifests.

In [ ]:
# ==============================================================================
# CELL 4.1: G4 MASTER FORGE V2.1 (NATIVE KERAS 3 + PROCEDURAL CONFIG)
# RUNTIME: G4 / L4 (RTX 6000 Pro)
# ==============================================================================
import os
import gc
import json
import tensorflow as tf
import keras
from keras import layers, ops, regularizers
import yfinance as yf
import pandas as pd
import numpy as np
from sklearn.preprocessing import RobustScaler

# 1. ENVIRONMENT & HARDWARE
os.environ["KERAS_BACKEND"] = "tensorflow"
os.environ["TF_GPU_ALLOCATOR"] = "cuda_malloc_async"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

keras.backend.clear_session()
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
    keras.mixed_precision.set_global_policy('mixed_bfloat16')
    print(f"[SYSTEM] Forge Ready: {gpus[0].name}")

# 2. TARGET DIRECTORY
DRIVE_DIR = "/content/drive/MyDrive/Bot_Investitii_Models"
os.makedirs(DRIVE_DIR, exist_ok=True)

# 3. CORE UTILITIES
@keras.saving.register_keras_serializable()
def sharpe_loss(y_true, y_pred):
    y_pred = ops.cast(y_pred, 'float32')
    y_true = ops.cast(y_true, 'float32')
    r = y_pred * y_true
    return -(ops.mean(r) / (ops.std(r) + 1e-9))

def fetch_forge_data(symbol):
    start = pd.to_datetime("today") - pd.DateOffset(days=700)
    tmp_df = yf.download(symbol, start=start, interval="1h", progress=False)
    if isinstance(tmp_df.columns, pd.MultiIndex): tmp_df.columns = tmp_df.columns.get_level_values(0)
    tmp_df = tmp_df.reset_index()
    tmp_df['Date'] = pd.to_datetime(tmp_df['Datetime' if 'Datetime' in tmp_df.columns else 'Date']).dt.tz_localize(None)
    tmp_df.set_index('Date', inplace=True)
    tmp_df['Log_Returns'] = np.log(tmp_df['Close'] / tmp_df['Close'].shift(1))
    tmp_df['Target'] = (tmp_df['Close'].shift(-6) / tmp_df['Close']) - 1.0
    delta = tmp_df['Close'].diff()
    gain = delta.where(delta > 0, 0).rolling(14).mean()
    loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
    tmp_df['RSI'] = 100 - 100 / (1 + gain / loss.replace(0, 1e-9))
    tmp_df['Vol_20'] = tmp_df['Log_Returns'].rolling(20).std()
    features = ['Log_Returns', 'RSI', 'Vol_20']
    tmp_df.dropna(subset=features + ['Target']).to_parquet(f"{symbol}_forge_data.parquet")
    return features

def build_temporal_fusion_engine(input_shape, dropout_rate):
    inputs = keras.Input(shape=input_shape)
    x = layers.BatchNormalization()(inputs)
    gate = layers.Dense(input_shape[-1], activation='sigmoid')(x)
    x = layers.Multiply()([x, gate])
    x = layers.Dense(128, activation='gelu')(x)

    x = layers.Bidirectional(layers.LSTM(64, return_sequences=True))(x)
    x = layers.LayerNormalization()(layers.Dropout(dropout_rate)(x))

    attn = layers.MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.1)(x, x)
    x = layers.LayerNormalization()(layers.Add()([x, attn]))

    x = layers.GlobalAveragePooling1D()(x)
    x = layers.Dense(64, activation='gelu', kernel_regularizer=regularizers.L2(1e-4))(x)
    outputs = layers.Dense(1, activation='tanh', dtype='float32')(x)
    return keras.Model(inputs, outputs)

def execute_g4_forge(symbol, params):
    print(f"\n[FORGE] Initiating: {symbol}")

    # Save Manifest (Hyperparameters)
    config_path = f"{DRIVE_DIR}/Keras3_Best_{symbol}_config.json"
    with open(config_path, 'w') as f:
        json.dump(params, f, indent=4)

    features = fetch_forge_data(symbol)
    df = pd.read_parquet(f"{symbol}_forge_data.parquet")
    idx = int(len(df) * 0.8)

    scaler = RobustScaler()
    X = scaler.fit_transform(df[features].iloc[:idx]).astype(np.float32)
    y = df['Target'].iloc[:idx].values.astype(np.float32)
    X_val = scaler.transform(df[features].iloc[idx:]).astype(np.float32)
    y_val = df['Target'].iloc[idx:].values.astype(np.float32)

    # Simple Windowing
    def window(X, y, ts=72):
        return np.array([X[i:i+ts] for i in range(len(X)-ts+1)]), np.array(y[ts-1:])

    X_w, y_w = window(X, y)
    X_v, y_v = window(X_val, y_val)

    model = build_temporal_fusion_engine((72, 3), dropout_rate=params['dropout'])
    model.compile(optimizer=keras.optimizers.Adam(learning_rate=params['lr']), loss=sharpe_loss, jit_compile=True)

    ckpt = keras.callbacks.ModelCheckpoint(f"{DRIVE_DIR}/Keras3_Best_{symbol}.weights.h5", save_best_only=True, save_weights_only=True)
    early = keras.callbacks.EarlyStopping(patience=10, restore_best_weights=True)

    model.fit(X_w, y_w, validation_data=(X_v, y_v), epochs=50, batch_size=params['batch'], callbacks=[ckpt, early], verbose=0)
    print(f"[SUCCESS] {symbol} forged and secured in Drive.")

# 4. CONFIGURATION MANIFEST
TICKERS_CONFIG = {
    "AAPL":    {"lr": 0.0005, "batch": 1024, "dropout": 0.3},
    "BTC-USD": {"lr": 0.0008, "batch": 512,  "dropout": 0.2},
    "ETH-USD": {"lr": 0.0008, "batch": 512,  "dropout": 0.2},
    "GOOGL":   {"lr": 0.0004, "batch": 1024, "dropout": 0.3},
    "MSFT":    {"lr": 0.0004, "batch": 1024, "dropout": 0.3},
    "NVDA":    {"lr": 0.0006, "batch": 512,  "dropout": 0.25},
    "SPY":     {"lr": 0.0003, "batch": 2048, "dropout": 0.15}
}

for ticker, params in TICKERS_CONFIG.items():
    try: execute_g4_forge(ticker, params)
    except Exception as e: print(f"[ERROR] {ticker}: {e}")

## Phase 5: Institutional Inference Engine (Streamlit)

Purpose: A decoupled inference application. It reconstructs the model architecture from JSON manifests and executes lightning-fast vector predictions without the overhead of a training loop.

In [ ]:
%%writefile app.py
# ==============================================================================
# CELL 5
# STREAMLIT BACKTEST ENGINE
# RUNTIME: General (Writes to local disk)
# PURPOSE: Unified Global Glassmorphism with CUDA Crash Immunity
# ==============================================================================
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

import streamlit as st
import numpy as np
import pandas as pd
import yfinance as yf
from sklearn.preprocessing import RobustScaler
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import (
    Dense, Input, Bidirectional, Conv1D, GRU, LSTM,
    BatchNormalization, LayerNormalization, Add,
    MultiHeadAttention, GlobalAveragePooling1D, Multiply, Lambda, Reshape, Concatenate
)
import tensorflow.keras.backend as K
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.regularizers import l2
import warnings
import gc

tf.random.set_seed(42)
np.random.seed(42)

for gpu in tf.config.list_physical_devices('GPU'):
    try: tf.config.experimental.set_memory_growth(gpu, True)
    except RuntimeError: pass
warnings.filterwarnings("ignore")

st.set_page_config(page_title="Institutional Quant AI v5.9", layout="wide", initial_sidebar_state="expanded")

st.markdown("""
<style>
    header[data-testid="stHeader"] {background: transparent !important;}

    [data-testid="stMetric"] {
        backdrop-filter: blur(12px);
        -webkit-backdrop-filter: blur(12px);
        border-radius: 12px;
        padding: 20px;
    }

    @media (prefers-color-scheme: dark) {
        .stApp {background: linear-gradient(135deg, #0b132b, #1c2541, #3a506b); color: #fff;}
        [data-testid="stSidebar"] > div:first-child {
            background: rgba(28, 37, 65, 0.4) !important;
            backdrop-filter: blur(15px) !important;
            -webkit-backdrop-filter: blur(15px) !important;
            border-right: 1px solid rgba(255, 255, 255, 0.1);
        }
        [data-testid="stMetric"] {
            background: rgba(255, 255, 255, 0.05);
            border: 1px solid rgba(255, 255, 255, 0.1);
        }
        [data-testid="stMetricValue"] {color: #5bc0be !important;}

        .stButton>button {
            background-color: rgba(91, 192, 190, 0.2);
            color: #5bc0be;
            border: 1px solid #5bc0be;
        }
        .stButton>button:hover {
            background-color: #5bc0be;
            color: #0b132b;
        }

        .stSelectbox div[data-baseweb="select"] > div,
        .stNumberInput div[data-baseweb="input"] > div {
            background-color: rgba(255, 255, 255, 0.05) !important;
            border: 1px solid rgba(255, 255, 255, 0.2) !important;
            color: #ffffff !important;
        }
        .stSelectbox svg { fill: #ffffff !important; }
        .stNumberInput input { color: #ffffff !important; }
    }

    @media (prefers-color-scheme: light) {
        .stApp {background: linear-gradient(135deg, #f0f4f8, #d9e2ec, #bcccdc); color: #102a43;}
        [data-testid="stSidebar"] > div:first-child {
            background: rgba(255, 255, 255, 0.5) !important;
            backdrop-filter: blur(20px) !important;
            -webkit-backdrop-filter: blur(20px) !important;
            border-right: 1px solid rgba(0, 0, 0, 0.05);
        }
        [data-testid="stMetric"] {
            background: rgba(255, 255, 255, 0.6);
            border: 1px solid rgba(255, 255, 255, 0.8);
            box-shadow: 0 4px 6px rgba(0, 0, 0, 0.05);
        }
        [data-testid="stMetricValue"] {color: #0b69a3 !important;}

        .stButton>button {
            background-color: rgba(11, 105, 163, 0.1);
            color: #0b69a3;
            border: 1px solid #0b69a3;
        }
        .stButton>button:hover {
            background-color: #0b69a3;
            color: #ffffff;
        }

        .stSelectbox div[data-baseweb="select"] > div,
        .stNumberInput div[data-baseweb="input"] > div {
            background-color: rgba(255, 255, 255, 0.4) !important;
            border: 1px solid rgba(0, 0, 0, 0.1) !important;
            color: #102a43 !important;
        }
        .stSelectbox svg { fill: #102a43 !important; }
        .stNumberInput input { color: #102a43 !important; }
    }
</style>
""", unsafe_allow_html=True)

st.title("Institutional Quant AI v5.9 - Microstructure Engine")

ARCHS = ["TCN + MHA + GRU (Sniper Alpha v2)", "BiLSTM + Attention (Deep Memory v2)", "G4 Heavy Transformer (VRAM Intensive)"]
IDEAL_CONFIGS = {
    "AAPL": {"arch": "G4 Heavy Transformer (VRAM Intensive)", "time_step": 72, "ensemble_size": 15, "mc_passes": 100, "cost_bps": 8, "target_vol": 60, "threshold": 0.05, "lr": 0.0005, "epochs": 100, "batch": 1024},
    "BTC-USD": {"arch": "G4 Heavy Transformer (VRAM Intensive)", "time_step": 168, "ensemble_size": 15, "mc_passes": 100, "cost_bps": 18, "target_vol": 80, "threshold": 0.1, "lr": 0.0005, "epochs": 100, "batch": 1024}
}

if "last_ticker" not in st.session_state: st.session_state.last_ticker = None
def apply_config(ticker: str):
    c = IDEAL_CONFIGS.get(ticker, IDEAL_CONFIGS["AAPL"])
    st.session_state["cfg_arch"] = c["arch"]; st.session_state["cfg_time_step"] = c["time_step"]
    st.session_state["cfg_ensemble"] = c["ensemble_size"]; st.session_state["cfg_mc"] = c["mc_passes"]
    st.session_state["cfg_cost"] = c["cost_bps"]; st.session_state["cfg_vol"] = c["target_vol"]
    st.session_state["cfg_thresh"] = float(c["threshold"]); st.session_state["cfg_lr"] = float(c["lr"])
    st.session_state["cfg_epochs"] = int(c["epochs"]); st.session_state["cfg_batch"] = int(c["batch"])

with st.sidebar:
    st.header("1. Asset Class")
    ticker = st.selectbox("Select Asset", ["AAPL", "NVDA", "MSFT", "GOOGL", "SPY", "BTC-USD", "ETH-USD"])
    manual_override = st.checkbox("Enable Manual Override", value=False)
    if ticker != st.session_state.last_ticker and not manual_override:
        apply_config(ticker); st.session_state.last_ticker = ticker

    st.header("2. Neural Controls")
    model_arch = st.selectbox("Architecture:", ARCHS, index=ARCHS.index(st.session_state.get("cfg_arch", ARCHS[0])))
    time_step = st.slider("Lookback window (Hours)", 24, 168, step=24, value=st.session_state.get("cfg_time_step", 72))
    ensemble_size = st.slider("Ensemble members", 1, 50, value=st.session_state.get("cfg_ensemble", 15))
    mc_passes = st.slider("MC Dropout passes", 10, 500, value=st.session_state.get("cfg_mc", 100))
    epochs_count = st.slider("Max Epochs", 10, 200, value=st.session_state.get("cfg_epochs", 100))

    batch_opts = [64, 128, 256, 512, 1024, 2048]
    curr_batch = st.session_state.get("cfg_batch", 1024)
    batch_size = st.selectbox("Batch Size", batch_opts, index=batch_opts.index(curr_batch if curr_batch in batch_opts else 1024))
    learning_rate = st.number_input("Learning Rate", value=st.session_state.get("cfg_lr", 0.0005), format="%.5f")

    st.header("3. Date Range")
    max_history = pd.to_datetime("today") - pd.DateOffset(days=700)
    start = st.date_input("Start Date", value=max_history.date(), min_value=max_history.date())
    end   = st.date_input("End Date", value=pd.to_datetime("today").date())

    st.header("4. Risk Management")
    long_only = st.checkbox("Long Only Mode", value=True)
    cost_bps = st.slider("Cost (bps)", 0, 50, value=st.session_state.get("cfg_cost", 8))
    target_vol = st.slider("Target Volatility (%)", 10, 100, value=st.session_state.get("cfg_vol", 50))
    conviction_thresh = st.slider("Conviction Threshold", 0.0, 1.0, value=st.session_state.get("cfg_thresh", 0.1))
    use_kelly = st.checkbox("Kelly Criterion sizing", value=True)

class StableGaussianNoise(tf.keras.layers.Layer):
    def __init__(self, stddev, **kwargs):
        super(StableGaussianNoise, self).__init__(**kwargs)
        self.stddev = stddev
    def call(self, inputs, training=None):
        if training:
            return inputs + tf.random.normal(tf.shape(inputs), mean=0.0, stddev=self.stddev, dtype=inputs.dtype)
        return inputs

class StableDropout(tf.keras.layers.Layer):
    def __init__(self, rate, **kwargs):
        super(StableDropout, self).__init__(**kwargs)
        self.rate = rate
    def call(self, inputs, training=None):
        if training:
            return tf.nn.dropout(inputs, rate=self.rate)
        return inputs

def sharpe_loss(y_true, y_pred):
    strategy_returns = y_pred * y_true
    mean_return = K.mean(strategy_returns)
    std_return = K.std(strategy_returns) + 1e-9
    return -(mean_return / std_return)

class Time2Vec(tf.keras.layers.Layer):
    def __init__(self, kernel_size=1, **kwargs):
        super(Time2Vec, self).__init__(**kwargs)
        self.k = kernel_size
    def build(self, input_shape):
        self.wb = self.add_weight(name='wb', shape=(input_shape[-1],), initializer='uniform', trainable=True)
        self.bb = self.add_weight(name='bb', shape=(input_shape[-1],), initializer='uniform', trainable=True)
        self.wa = self.add_weight(name='wa', shape=(1, input_shape[-1], self.k), initializer='uniform', trainable=True)
        self.ba = self.add_weight(name='ba', shape=(1, input_shape[-1], self.k), initializer='uniform', trainable=True)
        super(Time2Vec, self).build(input_shape)
    def call(self, inputs):
        bias = self.wb * inputs + self.bb
        dp = K.dot(inputs, self.wa) + self.ba
        ret = tf.concat([tf.expand_dims(bias, -1), tf.math.sin(dp)], -1)
        return tf.reshape(ret, (-1, tf.shape(inputs)[1], ret.shape[2] * ret.shape[3]))

class QuantPipeline:
    def __init__(self, symbol, start_date, end_date, time_step, model_architecture):
        self.symbol = symbol; self.start_date = start_date; self.end_date = end_date
        self.time_step = time_step; self.model_architecture = model_architecture; self.scaler = RobustScaler()

    def fetch_and_engineer(self):
        df = yf.download(self.symbol, start=self.start_date, end=self.end_date, interval="1h", progress=False)
        m_df = yf.download(["^VIX", "^TNX", "SPY", "BTC-USD"], start=self.start_date, end=self.end_date, interval="1h", progress=False)

        if isinstance(m_df.columns, pd.MultiIndex): m_df = m_df['Close']
        m_df.rename(columns={'^VIX': 'VIX', '^TNX': 'TNX'}, inplace=True)
        m_df.ffill(inplace=True)

        m_df['SPY_Ret'] = np.log(m_df['SPY'] / m_df['SPY'].shift(1))
        m_df['BTC_Ret'] = np.log(m_df['BTC-USD'] / m_df['BTC-USD'].shift(1))
        m_df.drop(columns=['SPY', 'BTC-USD'], inplace=True, errors='ignore')

        if type(m_df.index) == pd.DatetimeIndex: m_df.index = m_df.index.tz_localize(None)
        if 'Date' not in df.columns: df = df.reset_index()
        df.rename(columns={'Datetime':'Date', 'date':'Date'}, inplace=True, errors='ignore')
        df['Date'] = pd.to_datetime(df['Date']).dt.tz_localize(None)
        if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)

        df = pd.merge(df, m_df, left_on='Date', right_index=True, how='left')
        df = df.ffill().bfill().fillna(0.0)

        df['Sentiment_Score'] = 0.0; df.set_index('Date', inplace=True)
        c, h, l, o = df['Close'], df['High'], df['Low'], df['Open']
        df['Log_Returns'] = np.log(c / c.shift(1))
        df['Target'] = (df['Close'].shift(-6) / df['Close']) - 1.0
        df['Step_Return'] = (df['Close'].shift(-1) / df['Close']) - 1.0

        df['EMA_20'] = c.ewm(span=20).mean(); df['SMA_50'] = c.rolling(50).mean(); df['Macro_Bull'] = df['EMA_20'] > df['SMA_50']
        df['Garman_Klass'] = ((0.5 * np.log(h/l)**2 - (2*np.log(2)-1) * np.log(c/o)**2).clip(lower=0)).ewm(span=72, adjust=False).mean()

        delta = c.diff()
        gain = delta.where(delta > 0, 0).rolling(14).mean()
        loss = (-delta.where(delta < 0, 0)).rolling(14).mean()
        df['RSI'] = 100 - 100 / (1 + gain / loss.replace(0, 1e-9))
        df['MACD'] = c.ewm(span=12).mean() - c.ewm(span=26).mean()

        ann = 8760 if self.symbol in ["BTC-USD", "ETH-USD"] else 1638
        df['Vol_Regime'] = df['Log_Returns'].rolling(20).std() * np.sqrt(ann)

        t = df.index; df['RTH_Active'] = ((t.dayofweek < 5) & (t.hour >= 9) & (t.hour <= 16)).astype(float)

        features = ['Log_Returns', 'RSI', 'MACD', 'Vol_Regime', 'VIX', 'TNX', 'SPY_Ret', 'BTC_Ret', 'Sentiment_Score', 'RTH_Active']
        df.dropna(subset=features, inplace=True)
        return df

    def prepare_tensors(self, df):
        feature_cols = ['Sentiment_Score', 'Log_Returns','Garman_Klass','RSI','MACD','Vol_Regime','VIX', 'TNX', 'SPY_Ret', 'BTC_Ret', 'RTH_Active']
        idx = int(len(df) * 0.8)
        tr_df, te_df = df.iloc[:idx], df.iloc[idx:]

        if len(te_df) <= self.time_step:
            raise ValueError(f"Selected date range is too short. The testing set contains {len(te_df)} rows, but your Lookback Window requires {self.time_step} rows. Please select an older Start Date.")

        X_tr_sc = self.scaler.fit_transform(tr_df[feature_cols]).astype(np.float32)
        X_te_sc = self.scaler.transform(te_df[feature_cols]).astype(np.float32)
        y_tr, y_te = tr_df['Target'].values.astype(np.float32), te_df['Target'].values.astype(np.float32)

        def window(X, y, ts):
            return np.array([X[i:i+ts] for i in range(len(X)-ts+1)]), np.array(y[ts-1:])

        X_tr, y_tr = window(X_tr_sc, y_tr, self.time_step)
        X_te, y_te = window(X_te_sc, y_te, self.time_step)

        valid_tr = ~np.isnan(y_tr)
        X_tr_train, y_tr_train = X_tr[valid_tr], y_tr[valid_tr]

        valid_te = ~np.isnan(y_te)
        X_te_eval, y_te_eval = X_te[valid_te], y_te[valid_te]

        return X_tr_train, y_tr_train, X_te_eval, y_te_eval, X_te, y_te, te_df.iloc[self.time_step-1 : ]

    def build_model(self, input_shape, dropout_rate=0.35):
        inputs = Input(shape=input_shape)
        x = StableGaussianNoise(0.005)(inputs)
        x = BatchNormalization()(x)
        time_embedding = Time2Vec(kernel_size=4)(x)
        x = Concatenate()([x, time_embedding])

        if self.model_architecture == "TCN + MHA + GRU (Sniper Alpha v2)":
            c1 = Conv1D(64, 3, dilation_rate=1, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(x)
            c8 = Conv1D(64, 3, dilation_rate=8, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(
                 Conv1D(64, 3, dilation_rate=4, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(
                 Conv1D(64, 3, dilation_rate=2, padding='causal', activation='gelu', kernel_regularizer=l2(1e-4))(c1)))
            tcn = LayerNormalization()(Add()([c1, c8]))
            attn_out = MultiHeadAttention(num_heads=4, key_dim=32, dropout=0.0)(tcn, tcn)
            attn = LayerNormalization()(Add()([tcn, StableDropout(0.1)(attn_out)]))
            x = Bidirectional(GRU(128, return_sequences=False))(attn)
            x = StableDropout(dropout_rate)(x)

        elif self.model_architecture == "BiLSTM + Attention (Deep Memory v2)":
            x = Bidirectional(LSTM(128, return_sequences=True))(x)
            x = LayerNormalization()(StableDropout(dropout_rate)(x))
            x = Bidirectional(LSTM(64, return_sequences=True))(x)
            x = LayerNormalization()(StableDropout(dropout_rate)(x))
            score = Dense(1)(x)
            weights_attn = tf.keras.layers.Softmax(axis=1)(score)
            x = Lambda(lambda z: tf.reduce_sum(z, axis=1))(Multiply()([x, weights_attn]))

        else:
            x = Dense(128)(x)
            x_norm = LayerNormalization(epsilon=1e-6)(x)
            attn_out = MultiHeadAttention(num_heads=8, key_dim=64, dropout=0.0)(x_norm, x_norm)
            attn_1 = StableDropout(dropout_rate)(attn_out)
            x = Add()([x, attn_1])
            x_norm_2 = LayerNormalization(epsilon=1e-6)(x)
            ff_1 = Dense(128)(Dense(512, activation='gelu')(x_norm_2))
            x = Add()([x, ff_1])
            x = GlobalAveragePooling1D()(LayerNormalization(epsilon=1e-6)(x))
            x = StableDropout(dropout_rate)(x)

        x = Dense(128, activation='gelu', kernel_regularizer=l2(1e-4))(x)
        x = StableDropout(0.3)(x)
        outputs = Dense(1, activation='tanh', dtype='float32')(x)
        return Model(inputs, outputs)

if st.button(f"Execute Pipeline - {ticker}"):
    pipeline = QuantPipeline(ticker, start, end, time_step, model_arch)
    with st.status(f"Running {model_arch} on {ticker}...", expanded=True):
        try:
            df = pipeline.fetch_and_engineer()
            X_tr_train, y_tr_train, X_te_eval, y_te_eval, X_te, y_te, test_ctx = pipeline.prepare_tensors(df)
        except Exception as e:
            st.error(f"Data Pipeline Error: {e}")
            st.stop()

        preds = []
        for i in range(ensemble_size):
            K.clear_session()
            gc.collect()

            tf.random.set_seed(42 + i)
            model = pipeline.build_model((X_tr_train.shape[1], X_tr_train.shape[2]))
            model.compile(optimizer=Adam(learning_rate=learning_rate, clipnorm=1.0), loss=sharpe_loss)

            hist = model.fit(X_tr_train, y_tr_train, validation_data=(X_te_eval, y_te_eval), epochs=epochs_count, batch_size=batch_size, callbacks=[EarlyStopping(patience=15, restore_best_weights=True)], verbose=0)

            ds = tf.data.Dataset.from_tensor_slices(X_te).batch(batch_size)
            p_ensemble = np.mean([np.concatenate([model(b, training=True).numpy().flatten() for b in ds]) for _ in range(mc_passes)], axis=0)
            preds.append(p_ensemble)

            del model
            del hist

        avg_preds = np.mean(preds, axis=0)
        avg_uncert = np.std(preds, axis=0)

    bt = test_ctx.copy()
    bt['Predicted_6H_Return'] = avg_preds
    bt['Uncertainty'] = avg_uncert
    bt['Step_Return'] = bt['Step_Return'].fillna(0.0)
    bt['Actual_6H_Return'] = y_te

    p_mean = bt['Predicted_6H_Return'].rolling(168, min_periods=2).mean().fillna(bt['Predicted_6H_Return'])
    p_std  = bt['Predicted_6H_Return'].rolling(168, min_periods=2).std().replace(0, 1e-9).fillna(1.0)
    p_z    = (bt['Predicted_6H_Return'] - p_mean) / p_std

    aligned = (bt['Macro_Bull'] & (p_z > 0)) | (~bt['Macro_Bull'] & (p_z < 0))
    bt['Conviction'] = (tf.math.sigmoid(p_z * np.where(aligned, 2.5, 1.0)).numpy() - 0.5) * 2
    bt['Conviction'] *= (1 - 0.5 * (bt['Uncertainty'] / (bt['Uncertainty'].cummax() + 1e-9)))
    bt['Conviction'] = np.where(bt['Conviction'].abs() >= np.where(aligned, conviction_thresh*0.3, conviction_thresh), bt['Conviction'], 0.0)

    if use_kelly:
        correct = (np.sign(bt['Predicted_6H_Return']) == np.sign(bt['Actual_6H_Return'])).astype(float)
        kp = correct.shift(6).rolling(60).mean().fillna(0.5).clip(0.01, 0.99)
        pr = bt['Actual_6H_Return'].where(bt['Actual_6H_Return']>0, np.nan).shift(6).rolling(60).mean().fillna(0.001)
        nr = bt['Actual_6H_Return'].where(bt['Actual_6H_Return']<0, np.nan).abs().shift(6).rolling(60).mean().fillna(0.001)
        kf = (0.5 * (kp - (1 - kp) / (pr/nr.replace(0,1e-9)).clip(0.1,5))).clip(0,1)
    else:
        kf = 1.0

    ann = 8760 if ticker in ["BTC-USD", "ETH-USD"] else 1638
    safe_gk = bt['Garman_Klass'].replace(0, 1e-9)
    vol_scal = (target_vol / ((np.sqrt(safe_gk) * np.sqrt(ann)) * 100)).clip(0.2, 3.0)

    bt['Position']     = (bt['Conviction'] * vol_scal * kf).clip(0 if long_only else -3, 3).fillna(0)
    bt['Net_Return']   = (bt['Position'] * bt['Step_Return']) - (bt['Position'].diff().abs().fillna(0) * (cost_bps/10000))
    bt['Cum_Strategy'] = (1+bt['Net_Return']).cumprod()
    bt['Cum_Market']   = (1+bt['Step_Return']).cumprod()

    st.line_chart(bt[['Cum_Strategy','Cum_Market']])

    st.divider()
    st.subheader("Institutional Performance Metrics")

    total_net_return = (bt['Cum_Strategy'].iloc[-1] - 1) * 100
    win_rate         = (bt['Net_Return'] > 0).mean() * 100
    cum_max          = bt['Cum_Strategy'].cummax()
    drawdown         = (bt['Cum_Strategy'] - cum_max) / cum_max
    max_drawdown     = drawdown.min() * 100
    sharpe           = (bt['Net_Return'].mean() / (bt['Net_Return'].std() + 1e-12)) * np.sqrt(ann)
    sortino          = (bt['Net_Return'].mean() / (bt['Net_Return'][bt['Net_Return'] < 0].std() + 1e-12)) * np.sqrt(ann)
    model_accuracy   = (np.sign(bt['Predicted_6H_Return']) == np.sign(bt['Actual_6H_Return'])).mean() * 100

    col1, col2, col3 = st.columns(3)
    col1.metric("Net Total Return", f"{total_net_return:.2f}%")
    col2.metric("Win Rate",         f"{win_rate:.2f}%")
    col3.metric("Model Accuracy",   f"{model_accuracy:.2f}%")

    col4, col5, col6 = st.columns(3)
    col4.metric("Sharpe Ratio",  f"{sharpe:.2f}")
    col5.metric("Sortino Ratio", f"{sortino:.2f}")
    col6.metric("Max Drawdown",  f"{max_drawdown:.2f}%")

## Phase 6: Secure Deployment Tunnel

Purpose: Exposing the local Streamlit port to a public Cloudflare endpoint for real-time dashboarding.

In [ ]:
# ==============================================================================
# CELL 6: SECURE CLOUDFLARE TUNNEL
# RUNTIME: General (Run globally after Streamlit Cell)
# PURPOSE: Exposes the internal Streamlit port (8501) to a secure public URL.
# ==============================================================================
import os
import time
import subprocess
import re

print("1. Purging zombie processes and old tunnels...")
os.system("fuser -k 8501/tcp >/dev/null 2>&1")
os.system("pkill -f streamlit")
os.system("pkill -f cloudflared")
time.sleep(2)

print("2. Downloading Cloudflare Enterprise Tunnel...")
if not os.path.exists("cloudflared-linux-amd64"):
    os.system("wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64")
    os.system("chmod +x cloudflared-linux-amd64")

print("3. Booting Streamlit server...")
os.system("nohup streamlit run app.py --server.headless true --server.address 0.0.0.0 > streamlit.log 2>&1 &")
time.sleep(4)

print("4. Establishing direct Cloudflare connection...")
os.system("nohup ./cloudflared-linux-amd64 tunnel --url http://127.0.0.1:8501 > cloudflare.log 2>&1 &")
time.sleep(6)

try:
    with open("cloudflare.log", "r") as f:
        logs = f.read()
        url_match = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs)

        if url_match:
            print("==================================================================")
            print(f" DASHBOARD LIVE: Click here -> {url_match.group(0)}")
            print("==================================================================")
        else:
            print("Tunnel URL not found yet. Attempting one more read...")
            time.sleep(3)
            with open("cloudflare.log", "r") as f2:
                logs2 = f2.read()
                url_match2 = re.search(r"https://[a-zA-Z0-9-]+\.trycloudflare\.com", logs2)
                if url_match2:
                    print("==================================================================")
                    print(f" DASHBOARD LIVE: Click here -> {url_match2.group(0)}")
                    print("==================================================================")
                else:
                    print("ERROR: Could not fetch URL. Raw logs:")
                    print(logs2)
except Exception as e:
    print(f"CRITICAL ERROR reading Cloudflare logs: {e}")

## Appendix A: Hardware Diagnostics & Kernel Recovery
*Note: The following cells contain low-level system commands for CUDA debugging. They are used to force GPU redetection or flush unrecoverable VRAM handles during extended training sessions. Do not run these during standard execution.*

In [ ]:
import os
# CRITICAL: This must be set before ANY tensorflow import to fully hide the broken driver
os.environ["CUDA_VISIBLE_DEVICES"] = "-1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
os.environ["TF_USE_LEGACY_KERAS"] = "1"

import tensorflow as tf
# Verify isolation
physical_devices = tf.config.list_physical_devices('GPU')
print(f"[SYSTEM] Active GPUs detected: {len(physical_devices)}")
if len(physical_devices) == 0:
    print("[SYSTEM] Hardware Isolation Successful. Operating in pure CPU mode.")

import yfinance as yf
import pandas as pd
import numpy as np
import optuna
import gc
import warnings
from tensorflow.keras.models import Model
from tensorflow.keras.layers import *
import tensorflow.keras.backend as K
from tensorflow.keras.optimizers import Adam
from sklearn.preprocessing import RobustScaler

warnings.filterwarnings("ignore")
tf.config.run_functions_eagerly(True)
tf.keras.utils.set_random_seed(42)

DB_PATH = "sqlite:///g4_master_forge.db"

def sharpe_loss(y_true, y_pred):
    r = y_pred * y_true
    return -(K.mean(r) / (K.std(r) + 1e-9))

def fetch_forge_data(symbol="AAPL"):
    start = pd.to_datetime("today") - pd.DateOffset(days=700)
    df = yf.download(symbol, start=start, interval="1h", progress=False)
    if isinstance(df.columns, pd.MultiIndex): df.columns = df.columns.get_level_values(0)
    df = df.reset_index()
    df['Date'] = pd.to_datetime(df['Datetime' if 'Datetime' in df.columns else 'Date']).dt.tz_localize(None)
    df.set_index('Date', inplace=True)
    df['Log_Returns'] = np.log(df['Close'] / df['Close'].shift(1))
    df['Target'] = (df['Close'].shift(-6) / df['Close']) - 1.0
    feats = ['Log_Returns']
    df_clean = df.dropna(subset=feats + ['Target']).copy()
    df_clean.to_parquet(f"{symbol}_forge_data.parquet")
    return feats

def objective_wrapper(trial, symbol, features):
    tf.keras.utils.set_random_seed(trial.number)
    batch = trial.suggest_categorical("batch", [512, 1024])
    df_clean = pd.read_parquet(f"{symbol}_forge_data.parquet")
    idx = int(len(df_clean) * 0.8)
    tr_df, te_df = df_clean.iloc[:idx], df_clean.iloc[idx:]

    scaler = RobustScaler()
    X_tr = scaler.fit_transform(tr_df[features]).astype(np.float32)
    X_te = scaler.transform(te_df[features]).astype(np.float32)
    y_tr, y_te = tr_df['Target'].values.astype(np.float32), te_df['Target'].values.astype(np.float32)

    def window(X, y, ts):
        return np.array([X[i:i+ts] for i in range(len(X)-ts+1)]), np.array(y[ts-1:])

    X_tr_w, y_tr_w = window(X_tr, y_tr, 72)
    X_te_w, y_te_w = window(X_te, y_te, 72)

    # Force model build on CPU logical device
    with tf.device('/CPU:0'):
        inputs = Input(shape=(X_tr_w.shape[1], X_tr_w.shape[2]))
        x = LSTM(16, activation='tanh')(inputs)
        outputs = Dense(1, activation='tanh')(x)
        model = Model(inputs, outputs)
        model.compile(optimizer=Adam(1e-3), loss=sharpe_loss)

        try:
            history = model.fit(X_tr_w, y_tr_w, validation_data=(X_te_w, y_te_w), epochs=1, batch_size=batch, verbose=0)
            val_loss = min(history.history['val_loss'])
        except Exception as e:
            print(f"Trial Failed: {e}")
            val_loss = 999.0
        finally:
            del model
            K.clear_session()
            gc.collect()
    return val_loss

def forge_g4_hyperparameters(stock="AAPL"):
    if os.path.exists("g4_master_forge.db"): os.remove("g4_master_forge.db")
    features = fetch_forge_data(stock)
    print(f"Entering Forge for {stock} (Isolated CPU Mode)...")
    study = optuna.create_study(study_name=f"G4_{stock}", storage=DB_PATH, direction="minimize")
    study.optimize(lambda t: objective_wrapper(t, stock, features), n_trials=5, catch=(Exception,))
    try:
        print(f"Forge Complete. Best Loss: {study.best_value:.6f}")
    except:
        print("Forge trial execution failed.")

print("\n========================================================")
print(" INITIATING G4 MASTER FORGE (HARDWARE ISOLATION)")
print("========================================================\n")

forge_g4_hyperparameters("AAPL")

In [ ]:
# STEP 2: FORCE GPU REDETECTION
import os
import tensorflow as tf

# Clear any isolation flags
os.environ.pop("CUDA_VISIBLE_DEVICES", None)

# Re-check physical devices
gpus = tf.config.list_physical_devices('GPU')

if gpus:
    print(f"[SUCCESS] {len(gpus)} GPU(s) found: {gpus}")
    for gpu in gpus:
        tf.config.experimental.set_memory_growth(gpu, True)
else:
    print("[FAIL] TensorFlow still cannot see the GPU. Please go to 'Runtime' -> 'Restart session' and run this cell first.")

In [ ]:
import os
import tensorflow as tf
import tensorflow.keras.backend as K

# 1. Clean environment
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# 2. Reset Keras and TensorFlow
K.clear_session()

# 3. Hardware Re-Initialization
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print("\n[SUCCESS] RTX 6000 Handle Re-established")
        print(f"[ACTIVE] {gpus[0].name}")
    except RuntimeError as e:
        print(f"[INFO] Handle already locked: {e}")
else:
    print("\n[CRITICAL] GPU not found. Ensure Runtime is G4/L4 and Restart Session was performed.")

In [ ]:
# STEP 1: VERIFY HARDWARE AT OS LEVEL
!nvidia-smi

In [ ]:
import os
# 1. ENSURE NO ISOLATION VARIABLES EXIST
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# 2. INITIALIZE TENSORFLOW
import tensorflow as tf

# 3. VERIFY AND CONFIGURE
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n[SUCCESS] G4 MASTER FORGE ONLINE")
        print(f"[DEVICE] Active GPU: {gpus[0].name}")
        print(f"[VRAM] 96GB RTX 6000 Pro Detected")
    except RuntimeError as e: print(e)
else:
    print("\n[CRITICAL] GPU still not seen by TF. Ensure 'Runtime Type' is set to G4/L4 and you just RESTARTED the session.")

In [ ]:
import os
# 1. ENSURE NO ISOLATION VARIABLES EXIST
os.environ.pop("CUDA_VISIBLE_DEVICES", None)
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"

# 2. INITIALIZE TENSORFLOW
import tensorflow as tf

# 3. VERIFY AND CONFIGURE
gpus = tf.config.list_physical_devices('GPU')
if gpus:
    try:
        for gpu in gpus:
            tf.config.experimental.set_memory_growth(gpu, True)
        print(f"\n[SUCCESS] G4 MASTER FORGE ONLINE")
        print(f"[DEVICE] Active GPU: {gpus[0].name}")
        print(f"[VRAM] 96GB RTX 6000 Pro Detected")
    except RuntimeError as e: print(e)
else:
    print("\n[CRITICAL] GPU still not seen by TF. Ensure 'Runtime Type' is set to G4/L4 and you just RESTARTED the session.")

In [ ]:
# ==============================================================================
# RUNTIME: G4 / L4 (RTX 6000 Pro)
# PURPOSE: Secondary attempt to flush unrecoverable CUDA handles.
# ==============================================================================
import os
import time
import sys

print("!!! SECONDARY KERNEL REBIRTH INITIATED !!!")
print("Flushing all remaining system buffers and killing process...")

time.sleep(1)

# Closing all file handles and forcing a low-level exit
sys.stdout.flush()
sys.stderr.flush()
os._exit(0)
# Closing all file handles and forcing a low-level exit
sys.stdout.flush()
sys.stderr.flush()

# UNCOMMENT THE LINE BELOW ONLY IF VRAM IS DEADLOCKED
# os._exit(0)